# Create TrygFonden Awards from Official Donation API

Creates OpenAlex award rows from TrygFonden's public Projects and Donations database on tryghed.dk.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/trygfonden_to_s3.py` to download the official API response, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes all parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** `https://tryghed.dk/api/donations?templateName=Donation`  
**Corroborating sitemap:** `https://tryghed.dk/sitemap-donations.xml`  
**S3 location:** `s3a://openalex-ingest/awards/trygfonden/trygfonden_donations.parquet`  
**Local full-source validation on 2026-05-26:** 8,824 donation rows, 8,824 unique SharepointId values, 100% title/recipient/year/type/target-area/landing-url coverage, 8,806 rows with DKK amounts (99.8%), total DKK 3,123,474,280, years 2013-2025.

**TrygFonden funder:**
- funder_id: 4320324424
- display_name: "TrygFonden"
- country: DK

**Source-shape note:** TrygFonden donation pages currently expose an unrelated outer page H1 in static HTML. The correct project title is in the official API field `Title` and the detail overlay; therefore this notebook uses the first-party JSON API parquet, not scraped page headings.

**Mapping summary:** One OpenAlex award row per TrygFonden donation record. Native key is `trygfonden-{SharepointId}`. `amount` is the API `Amount`; `currency` is DKK when amount is present. `funding_type` is `research` only for source `Type = Forskningsprojekt`, otherwise `grant`. `lead_investigator` is organization-level: the recipient organization is stored in `lead_investigator.affiliation.name`; person fields are NULL because the source publishes organizations, not PIs.

## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.trygfonden_awards_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/trygfonden/trygfonden_donations.parquet`;

In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-26 found 8,824 rows.
SELECT COUNT(*) as total_trygfonden_donations
FROM openalex.awards.trygfonden_awards_raw;

In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.trygfonden_awards_raw;

In [ ]:
%sql
-- Sample raw TrygFonden data.
SELECT
    funder_award_id,
    sharepoint_id,
    display_name,
    recipient_name,
    amount,
    currency,
    published_year,
    type,
    target_area,
    focus_area,
    council,
    landing_page_url
FROM openalex.awards.trygfonden_awards_raw
LIMIT 10;

## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5.
SELECT column_name
FROM (DESCRIBE openalex.awards.trygfonden_awards_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|obligated|outlay|grant|award|donation|stotte|support';

In [ ]:
%sql
-- Confirm amount/date coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_with_amount,
    ROUND(MIN(TRY_CAST(amount AS DOUBLE)), 0) AS min_amount,
    ROUND(MAX(TRY_CAST(amount AS DOUBLE)), 0) AS max_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_amount_dkk,
    COUNT(start_date) AS rows_with_start_date,
    COUNT(end_date) AS rows_with_end_date,
    MIN(TRY_CAST(published_year AS INT)) AS min_year,
    MAX(TRY_CAST(published_year AS INT)) AS max_year
FROM openalex.awards.trygfonden_awards_raw;

In [ ]:
%sql
-- Native-key inspection. SharepointId is unique and is used in funder_award_id.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT sharepoint_id) AS distinct_sharepoint_ids,
    COUNT(DISTINCT source_id) AS distinct_api_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.trygfonden_awards_raw;

In [ ]:
%sql
-- Source taxonomy distributions.
SELECT type, target_area, focus_area, COUNT(*) AS cnt, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_dkk
FROM openalex.awards.trygfonden_awards_raw
GROUP BY type, target_area, focus_area
ORDER BY cnt DESC
LIMIT 50;

In [ ]:
%sql
-- Danish regional distribution from the source field `council`.
SELECT council, COUNT(*) AS cnt, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_dkk
FROM openalex.awards.trygfonden_awards_raw
GROUP BY council
ORDER BY cnt DESC;

## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
-- Must return exactly 1 row. If this is empty, STOP: the funder is missing from openalex.common.funder.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320324424;

## Step 2: Transform to OpenAlex Award Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.trygfonden_awards
USING delta
AS
WITH
trygfonden_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320324424
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        CASE WHEN TRY_CAST(amount AS DOUBLE) IS NOT NULL THEN 'DKK' ELSE NULL END AS parsed_currency,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(published_year AS INT) AS parsed_year,
        CONCAT_WS(
            ' - ',
            NULLIF(TRIM(target_area), ''),
            NULLIF(TRIM(focus_area), ''),
            NULLIF(TRIM(type), '')
        ) AS source_scheme
    FROM openalex.awards.trygfonden_awards_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,

        TRIM(g.display_name) as display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END as description,

        f.funder_id,
        g.native_award_id as funder_award_id,

        g.parsed_amount as amount,
        g.parsed_currency as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        CASE
            WHEN LOWER(TRIM(g.type)) = 'forskningsprojekt' THEN 'research'
            ELSE 'grant'
        END as funding_type,

        NULLIF(TRIM(g.source_scheme), '') as funder_scheme,

        'trygfonden_donations_api' as provenance,

        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_year) as start_year,
        COALESCE(YEAR(g.parsed_end_date), g.parsed_year) as end_year,

        struct(
            CAST(NULL AS STRING) as given_name,
            CAST(NULL AS STRING) as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.recipient_name), '') as name,
                CASE
                    WHEN g.council IN ('Syddanmark', 'Hovedstaden', 'Landsdækkende', 'Midtjylland', 'Sjælland', 'Nordjylland') THEN 'DK'
                    ELSE NULL
                END as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN trygfonden_funder f
)

SELECT * FROM awards_transformed;

## Step 3: Delete Previous Source Rows and Insert Priority 122

In [ ]:
%sql
-- Remove previous TrygFonden API data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'trygfonden_donations_api' AND priority = 122;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    122 as priority
FROM openalex.awards.trygfonden_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_trygfonden_awards
FROM openalex.awards.trygfonden_awards;

In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.trygfonden_awards;

In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_date,
    end_date,
    lead_investigator.affiliation.name AS recipient_org,
    lead_investigator.affiliation.country AS recipient_country,
    landing_page_url
FROM openalex.awards.trygfonden_awards
LIMIT 10;

In [ ]:
%sql
-- Check ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.trygfonden_awards;

In [ ]:
%sql
-- Check funder distribution. Should be only TrygFonden.
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.trygfonden_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;

In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(landing_page_url) as has_url,
    COUNT(lead_investigator.affiliation.name) as has_recipient_org,
    SUM(amount) as total_funding,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_with_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) as pct_description,
    ROUND(COUNT(lead_investigator.affiliation.name) * 100.0 / COUNT(*), 1) as pct_recipient_org
FROM openalex.awards.trygfonden_awards;

In [ ]:
%sql
-- Amount/currency fail-fast check for this grant pattern.
-- Local source validation found 99.8% amount coverage and only DKK.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS rows_with_positive_amount,
    ROUND(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_positive_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(CASE WHEN amount > 0 THEN amount END), 0) AS avg_positive_amount,
    ROUND(SUM(amount), 0) AS total_amount
FROM openalex.awards.trygfonden_awards;

In [ ]:
%sql
-- Year distribution.
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_dkk
FROM openalex.awards.trygfonden_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC;

In [ ]:
%sql
-- Funding type and scheme distribution.
SELECT funding_type, funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_dkk
FROM openalex.awards.trygfonden_awards
GROUP BY funding_type, funder_scheme
ORDER BY cnt DESC
LIMIT 50;

In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 122.
SELECT
    priority,
    provenance,
    COUNT(*) as cnt,
    COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids,
    ROUND(SUM(amount), 0) as total_dkk
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'trygfonden_donations_api' AND priority = 122
GROUP BY priority, provenance;